# 04 – MLflow Tracking & Model Serving
## ADSC21 Prüfungsleistung – Milwaukee Property Sales

**Ziel:** Experiment-Tracking mit MLflow, Modell-Registry und Diskussion der Produktivsetzung.
Entspricht dem Kursinhalt aus Sektion 5 (Notebooks 5A–5C: Model Comparison, Advanced Tracking, Model Serving).

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f'MLflow Version: {mlflow.__version__}')

## 1. Daten & Preprocessor

In [ ]:
df = pd.read_csv('../data/property_sales_clean.csv')
price_col = 'sale_price' if 'sale_price' in df.columns else 'saleprice'

numerical_features   = [c for c in ['fin_sqft', 'year_built', 'bedrooms', 'bathrooms', 'lotsize', 'stories'] if c in df.columns]
categorical_features = [c for c in ['style', 'extwall', 'zip_code'] if c in df.columns]

X = df[numerical_features + categorical_features]
y = df[price_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), numerical_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=10))]),
     categorical_features)
], remainder='drop')

print('Daten geladen.')

## 2. MLflow Experiment-Tracking

Wir loggen mehrere Modelle in einem MLflow-Experiment.
MLflow speichert Parameter, Metriken und das Modell-Artefakt.

In [ ]:
# MLflow Experiment anlegen
mlflow.set_tracking_uri('mlruns')  # lokaler Tracking Store
experiment_name = 'milwaukee_property_sales'
mlflow.set_experiment(experiment_name)
print(f'Experiment: {experiment_name}')

In [ ]:
def train_and_log(model, params, run_name):
    """Trainiert ein Modell und loggt alles in MLflow."""
    with mlflow.start_run(run_name=run_name):
        pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        mdape = np.median(np.abs((y_test - y_pred) / y_test)) * 100

        # Parameter loggen
        mlflow.log_params(params)

        # Metriken loggen
        mlflow.log_metrics({
            'rmse':  rmse,
            'mae':   mae,
            'r2':    r2,
            'mdape': mdape
        })

        # Modell als MLflow-Artefakt speichern
        mlflow.sklearn.log_model(
            pipe,
            artifact_path='model',
            registered_model_name=f'property_price_{run_name}'
        )

        print(f'[{run_name}] RMSE=${rmse:,.0f} | MAE=${mae:,.0f} | R²={r2:.4f} | MdAPE={mdape:.1f}%')
        return pipe, rmse

print('Funktion definiert.')

In [ ]:
# Experiment 1: Baseline
lr_pipe, lr_rmse = train_and_log(
    LinearRegression(),
    params={'model_type': 'LinearRegression'},
    run_name='baseline_lr'
)

# Experiment 2: Random Forest (Standard)
rf_pipe, rf_rmse = train_and_log(
    RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_split=5,
                          max_features='sqrt', random_state=42, n_jobs=-1),
    params={'model_type': 'RandomForest', 'n_estimators': 200,
            'max_depth': 20, 'min_samples_split': 5, 'max_features': 'sqrt'},
    run_name='random_forest'
)

# Experiment 3: Gradient Boosting
gb_pipe, gb_rmse = train_and_log(
    GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                               max_depth=4, subsample=0.8, random_state=42),
    params={'model_type': 'GradientBoosting', 'n_estimators': 300,
            'learning_rate': 0.05, 'max_depth': 4, 'subsample': 0.8},
    run_name='gradient_boosting'
)

## 3. Bestes Modell aus MLflow Registry laden & deployen

In [ ]:
# Bestes Modell anhand RMSE auswählen
best_pipe = min([(lr_pipe, lr_rmse), (rf_pipe, rf_rmse), (gb_pipe, gb_rmse)],
                key=lambda x: x[1])[0]
model_name = {lr_pipe: 'Linear Regression', rf_pipe: 'Random Forest',
              gb_pipe: 'Gradient Boosting'}[best_pipe]
print(f'Bestes Modell: {model_name}')

# Als Produktionsmodell speichern
joblib.dump(best_pipe, '../data/production_model.pkl')
print('Modell als production_model.pkl gespeichert.')

## 4. Inference-Beispiel (wie das Modell produktiv eingesetzt werden würde)

In [ ]:
# Beispiel-Anfrage (wie sie über eine REST-API ankommen würde)
example_input = pd.DataFrame([{
    'fin_sqft': 1200,
    'year_built': 1985,
    'bedrooms': 3,
    'bathrooms': 1,
    'lotsize': 5000,
    'stories': 1,
    'style': 'Ranch',
    'extwall': 'Brick',
    'zip_code': '53207'
}])

# Nur vorhandene Spalten
example_input = example_input[[c for c in example_input.columns if c in X.columns]]

prediction = best_pipe.predict(example_input)[0]
print(f'Vorhergesagter Verkaufspreis: ${prediction:,.0f}')

## 5. MLflow Model Serving – Konzept & Befehlsreferenz

Das Modell kann nach dem Training mit folgendem Befehl als REST-API gestartet werden:

```bash
# Modell aus MLflow Registry laden und als REST-Endpunkt starten
mlflow models serve \
  --model-uri 'models:/property_price_random_forest/Production' \
  --port 5001 \
  --no-conda
```

Dann kann der Endpunkt per HTTP aufgerufen werden:

```bash
curl -X POST http://localhost:5001/invocations \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_records": [{"fin_sqft": 1200, "year_built": 1985, "bedrooms": 3}]}'
```

### Produktivnahme: Wichtige Aspekte (vgl. Kurs Sektion 5)

1. **Datendrift-Monitoring:** Immobilienpreise ändern sich mit dem Markt. Regelmäßiges Retraining (z.B. jährlich bei neuen Milwaukee-Daten) ist notwendig.
2. **Reproduzierbarkeit:** MLflow sichert exakt welche Daten, welcher Code und welche Parameter zum Modell geführt haben.
3. **Feature-Stabilität:** ZIP-Codes und Gebäudestile können sich ändern. `handle_unknown='ignore'` im OneHotEncoder fängt neue Kategorien ab.
4. **Fairness:** ZIP-Codes als Feature können historische Diskriminierungsmuster verstärken – kritisch zu hinterfragen bei produktivem Einsatz.
5. **Skalierbarkeit:** Für höhere Last kann das MLflow-Modell in einen Container (Docker) verpackt und auf Kubernetes deployed werden.

In [ ]:
# MLflow UI starten (im Terminal ausführen)
print('MLflow UI starten mit: mlflow ui --port 5000')
print('Dann im Browser: http://localhost:5000')
print()

# Zusammenfassung der Runs ausgeben
client = mlflow.tracking.MlflowClient(tracking_uri='mlruns')
exp = client.get_experiment_by_name('milwaukee_property_sales')
if exp:
    runs = client.search_runs(exp.experiment_id, order_by=['metrics.rmse ASC'])
    print(f'Gespeicherte Runs ({len(runs)}):')
    for r in runs:
        print(f"  {r.info.run_name:25s} | RMSE=${r.data.metrics.get('rmse', 0):,.0f} | R²={r.data.metrics.get('r2', 0):.4f}")
else:
    print('MLflow Experiment noch nicht initialisiert. Erst Zellen oben ausführen.')